# 한자 운영 후보군 ChromaDB 비교 검증

기존 `src/graph/naming_graph.py`의 LangGraph 파이프라인과 내부 프롬프트 구조를 유지한 상태에서, 한자 검색 컬렉션만 테스트 전용 ChromaDB로 전환하여 운영 후보군의 적합성을 비교 검증합니다.

- 운영용 `data/chroma` 및 기존 파이프라인 파일은 수정하지 않습니다.
- 테스트 컬렉션은 로컬 `data/chroma_hanja_test` 경로에 준비된 ChromaDB를 사용합니다. 해당 DB 바이너리는 Git 커밋 대상에서 제외합니다.
- `hanja_col` 조회만 테스트 컬렉션으로 전환하며, `paper_col` 등 기타 컬렉션과 도구 호출 흐름은 기존 구조를 유지합니다.
- GPT 기반 최종 답변 비교가 필요한 경우 `RUN_LLM = True`로 설정한 뒤 실행합니다.


In [ ]:
# 1. 기본 환경 설정
# 테스트 DB 생성 시 사용한 임베딩 모델과 동일한 모델명을 사용하여 검색 조건의 일관성을 유지합니다.
from pathlib import Path
import hashlib
import importlib
import json
import os
import random
import re
import sys
import time
from datetime import datetime
from typing import Any

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

ROOT = Path.cwd()
if ROOT.name != "SKN29-3rd-4Team":
    ROOT = Path(r"C:\python-src\3rd_PROJECT\third_project\SKN29-3rd-4Team")

CHROMA_PATH = ROOT / "data" / "chroma_hanja_test"
RESULT_DIR = ROOT / "tests" / "results"

MODEL_NAME = "jhgan/ko-sroberta-multitask"
BASE_COLLECTION = "hanja_base_test_col"
CANDIDATE_COLLECTION = "hanja_candidate_test_col"
EXPANDED_COLLECTION = "hanja_expanded_test_col"

# OPENAI_API_KEY는 실행 환경에서 직접 입력하거나 환경변수로 주입합니다.
OPENAI_API_KEY = ""
RUN_LLM = True

# 비교 검증에 사용할 작명 질의입니다. 필요 시 동일 형식의 질의로 교체하여 재실행합니다.
QUESTION = "김(金)씨 성에 어울리는 여자 한자 이름 3개 추천해줘"

# 직접 검색 미리보기에서 사용할 기본 검색 결과 수입니다. naming_graph 내부 샘플링 수와는 별도로 관리됩니다.
N_RESULTS = 12

print("ROOT:", ROOT)
print("CHROMA_PATH:", CHROMA_PATH)
print("테스트 ChromaDB 준비 여부:", CHROMA_PATH.exists())


In [ ]:
# 2. API 키 설정 및 naming_graph 모듈 로드
# 프롬프트 원본은 naming_graph.py 내부 상수를 그대로 사용하여 운영 파이프라인과 동일성을 유지합니다.
# API 키가 없는 경우에도 구조 검증과 Chroma 연결 검증이 가능하도록 import 단계만 통과시킵니다.
real_api_key = OPENAI_API_KEY.strip() or os.getenv("OPENAI_API_KEY", "").strip()
if real_api_key:
    os.environ["OPENAI_API_KEY"] = real_api_key
else:
    os.environ.setdefault("OPENAI_API_KEY", "sk-test-not-used")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

import src.graph.naming_graph as naming_graph
naming_graph = importlib.reload(naming_graph)

print("naming_graph 로드 완료:", naming_graph.__file__)
print("GPT 실제 호출 가능 여부:", bool(real_api_key and real_api_key != "sk-test-not-used"))


In [ ]:
# 3. 테스트 ChromaDB 인덱싱 산출물 연결 검증
# 다음 세 컬렉션은 로컬 테스트 전용 ChromaDB에 준비되어 있어야 합니다.
embedding_fn = SentenceTransformerEmbeddingFunction(model_name=MODEL_NAME)
test_client = chromadb.PersistentClient(path=str(CHROMA_PATH))

EXPECTED_COUNTS = {
    BASE_COLLECTION: 2420,
    CANDIDATE_COLLECTION: 6564,
    EXPANDED_COLLECTION: 8984,
}

def get_test_collection(name: str):
    return test_client.get_collection(name=name, embedding_function=embedding_fn)

connection_report = {}
for collection_name, expected_count in EXPECTED_COUNTS.items():
    collection = get_test_collection(collection_name)
    actual_count = collection.count()
    connection_report[collection_name] = {
        "expected_count": expected_count,
        "actual_count": actual_count,
        "ok": actual_count == expected_count,
    }

print(json.dumps(connection_report, ensure_ascii=False, indent=2))
print("테스트 ChromaDB 경로:", CHROMA_PATH)


In [ ]:
# 4. 기존 내부 프롬프트 참조 위치 확인
# 프롬프트: 본 노트북은 신규 프롬프트를 정의하지 않고 naming_graph.py의 기존 프롬프트 상수를 사용합니다.
# 프롬프트 주입 지점: naming_graph.py 내부의 SystemMessage(content=...) 및 HumanMessage(content=...) 호출부입니다.
PROMPT_TARGETS = {
    "라우터 프롬프트": "_ROUTER_SYSTEM",
    "외자 이름 생성 프롬프트": "_GENERATE_SYSTEM_SINGLE",
    "순우리말 이름 생성 프롬프트": "_GENERATE_SYSTEM_KOREAN",
    "한자 이름 생성 프롬프트": "_GENERATE_SYSTEM",
    "한자 추천 검증 프롬프트": "_VERIFY_SYSTEM",
    "한자 추천 수정 프롬프트": "_REPAIR_SYSTEM",
    "발음 후보 생성 프롬프트": "_SOUND_GEN_SYSTEM",
}

for label, attr_name in PROMPT_TARGETS.items():
    prompt_text = getattr(naming_graph, attr_name, "")
    preview = str(prompt_text).replace("\n", " ")[:160]
    print(f"[{label}] {attr_name} / 길이={len(str(prompt_text))}")
    print(preview)
    print("-" * 80)


In [ ]:
# 5. 테스트 컬렉션 어댑터 구성
# 기존 파이프라인 파일은 수정하지 않고, 노트북 실행 범위에서만 rag_server의 hanja_col 조회 대상을 테스트 컬렉션으로 전환합니다.
# 한자 외 컬렉션은 기존 rag_server.search_rag 흐름을 유지합니다.
TEST_MODES = {
    "Baseline": [BASE_COLLECTION],
    "Expanded": [EXPANDED_COLLECTION],
    "Hybrid": [BASE_COLLECTION, CANDIDATE_COLLECTION],
}

ACTIVE_MODE = {"name": "Baseline"}

if not hasattr(naming_graph.rag_server, "_original_search_rag_for_notebook"):
    naming_graph.rag_server._original_search_rag_for_notebook = naming_graph.rag_server.search_rag
if not hasattr(naming_graph.rag_server, "_original_sample_hanja_for_notebook"):
    naming_graph.rag_server._original_sample_hanja_for_notebook = naming_graph.rag_server.sample_hanja
if not hasattr(naming_graph.rag_server, "_original_load_hanja_for_notebook"):
    naming_graph.rag_server._original_load_hanja_for_notebook = naming_graph.rag_server._load_person_name_hanja
if not hasattr(naming_graph.rag_server, "_original_get_ohaeng_for_notebook"):
    naming_graph.rag_server._original_get_ohaeng_for_notebook = naming_graph.rag_server.get_hanja_ohaeng

original_search_rag = naming_graph.rag_server._original_search_rag_for_notebook

def parse_hanja_where(query: str) -> dict[str, Any] | None:
    conditions: list[dict[str, Any]] = []
    stroke_match = re.search(r"(\d+)\s*획", query)
    if stroke_match:
        conditions.append({"strokes": int(stroke_match.group(1))})

    ohaeng_alias = {"木": "목", "火": "화", "土": "토", "金": "금", "水": "수"}
    for raw, normalized in {**ohaeng_alias, "목": "목", "화": "화", "토": "토", "금": "금", "수": "수"}.items():
        if f"{raw}오행" in query or f"{raw} 오행" in query:
            conditions.append({"resource_ohaeng": normalized})
            break

    if not conditions:
        return None
    if len(conditions) == 1:
        return conditions[0]
    return {"$and": conditions}

def query_hanja_test_collections(query: str, n_results: int = 10) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    where = parse_hanja_where(query)
    for collection_name in TEST_MODES[ACTIVE_MODE["name"]]:
        collection = get_test_collection(collection_name)
        result = collection.query(
            query_texts=[query],
            n_results=min(n_results, collection.count()),
            include=["documents", "metadatas", "distances"],
            **({"where": where} if where else {}),
        )
        ids = result.get("ids", [[]])[0]
        documents = result.get("documents", [[]])[0]
        metadatas = result.get("metadatas", [[]])[0]
        distances = result.get("distances", [[]])[0]
        for doc_id, document, metadata, distance in zip(ids, documents, metadatas, distances):
            rows.append({
                "id": doc_id,
                "document": document,
                "metadata": metadata or {},
                "distance": float(distance),
                "test_collection": collection_name,
            })
    rows.sort(key=lambda item: item["distance"])
    return rows[:n_results]

def format_hanja_rows(rows: list[dict[str, Any]]) -> str:
    if not rows:
        return "[hanja_col 테스트 결과 없음] 조건에 맞는 한자 문서를 찾지 못했습니다."
    lines = [f"[hanja_col 테스트 검색 결과] mode={ACTIVE_MODE['name']} / {len(rows)}건"]
    for index, row in enumerate(rows, 1):
        meta = row["metadata"]
        lines.append(
            f"[{index}] {row['document']}\n"
            f"    id={row['id']} / test_collection={row['test_collection']} / distance={row['distance']:.4f}\n"
            f"    hanja={meta.get('hanja')} / hangul={meta.get('hangul')} / 뜻={meta.get('sound_meaning')} / "
            f"획수={meta.get('strokes')} / 발음오행={meta.get('sound_ohaeng')} / 자원오행={meta.get('resource_ohaeng')} / source={meta.get('source')}"
        )
    return "\n".join(lines)

def test_search_rag(query: str, collection: str, n_results: int = 5) -> str:
    if collection != "hanja_col":
        return original_search_rag(query, collection, n_results=n_results)
    return format_hanja_rows(query_hanja_test_collections(query, n_results=n_results))

def load_test_person_name_hanja() -> list[tuple[str, dict[str, Any]]]:
    rows: list[tuple[str, dict[str, Any]]] = []
    seen: set[tuple[str, str, str]] = set()
    for collection_name in TEST_MODES[ACTIVE_MODE["name"]]:
        collection = get_test_collection(collection_name)
        result = collection.get(where={"is_person_name_hanja": True}, include=["documents", "metadatas"])
        for document, metadata in zip(result.get("documents", []), result.get("metadatas", [])):
            metadata = metadata or {}
            key = (str(metadata.get("hanja", "")), str(metadata.get("hangul", "")), str(metadata.get("profile_id", "")))
            if key in seen:
                continue
            seen.add(key)
            rows.append((document, metadata))
    return rows

def get_test_hanja_ohaeng(hanja_char: str) -> str:
    for _, metadata in load_test_person_name_hanja():
        if metadata.get("hanja") == hanja_char:
            return metadata.get("resource_ohaeng", "") or ""
    return ""

def sample_test_hanja(query: str, n_results: int = 20) -> str:
    pool = load_test_person_name_hanja()
    where = parse_hanja_where(query) or {}
    target_ohaeng = where.get("resource_ohaeng") if isinstance(where, dict) else None
    if target_ohaeng:
        pool = [(doc, meta) for doc, meta in pool if meta.get("resource_ohaeng") == target_ohaeng]
    if not pool:
        return f"[인명용 한자 후보 없음] mode={ACTIVE_MODE['name']} / query={query}"
    seed = int(hashlib.sha256(f"{ACTIVE_MODE['name']}::{query}".encode("utf-8")).hexdigest()[:8], 16)
    sampled = random.Random(seed).sample(pool, min(n_results, len(pool)))
    lines = [f"[인명용 한자 테스트 후보] mode={ACTIVE_MODE['name']} / {len(sampled)}건 / 전체 후보 {len(pool)}건"]
    for index, (_, meta) in enumerate(sampled, 1):
        lines.append(
            f"[{index}] {meta.get('hanja')}({meta.get('hangul')}) | "
            f"뜻: {meta.get('sound_meaning')} | 획수: {meta.get('strokes')} | "
            f"발음오행: {meta.get('sound_ohaeng')} | 자원오행: {meta.get('resource_ohaeng')} | source: {meta.get('source')}"
        )
    return "\n".join(lines)

def apply_test_adapter(mode: str) -> None:
    if mode not in TEST_MODES:
        raise ValueError(f"알 수 없는 테스트 모드입니다: {mode}")
    ACTIVE_MODE["name"] = mode
    naming_graph.rag_server.search_rag = test_search_rag
    naming_graph.rag_server.sample_hanja = sample_test_hanja
    naming_graph.rag_server._load_person_name_hanja = load_test_person_name_hanja
    naming_graph.rag_server.get_hanja_ohaeng = get_test_hanja_ohaeng
    if hasattr(naming_graph, "_hanja_stroke_cache"):
        naming_graph._hanja_stroke_cache = None

print("테스트 어댑터 준비 완료")


In [ ]:
# 6. 테스트 ChromaDB 연결 사전 검증
# GPT 호출 전 세 가지 비교 모드가 테스트 컬렉션과 정상 연결되는지 확인합니다.
preview_report = {}
for mode in TEST_MODES:
    apply_test_adapter(mode)
    rows = query_hanja_test_collections(QUESTION, n_results=3)
    preview_report[mode] = {
        "collection_names": TEST_MODES[mode],
        "retrieved_count": len(rows),
        "sample_ids": [row["id"] for row in rows],
        "sample_sources": [row["metadata"].get("source") for row in rows],
    }

print(json.dumps(preview_report, ensure_ascii=False, indent=2))


In [ ]:
# 7. naming_graph 실행 및 추적 함수 정의
# 프롬프트 입력 지점: app.stream(initial_state, ...) 실행 시 naming_graph.py 내부에서 기존 SystemMessage와 HumanMessage가 구성됩니다.
# initial_state 구조는 기존 파이프라인 입력 형식과 동일하게 유지합니다.
def build_initial_state(query: str) -> dict[str, Any]:
    return {
        "query": query,
        "context": "",
        "next_action": "generate",
        "answer": "",
        "iterations": 0,
        "used_tools": [],
        "collections": [],
        "name_length": 2,
        "surname_hanja": "",
    }

def compact_trace_value(value: Any) -> Any:
    if isinstance(value, str):
        return value[:300] + ("..." if len(value) > 300 else "")
    if isinstance(value, list):
        return value[:8]
    if isinstance(value, dict):
        return {key: compact_trace_value(inner) for key, inner in list(value.items())[:8]}
    return value

def summarize_update(node_name: str, update: dict[str, Any]) -> dict[str, Any]:
    return {
        "node": node_name,
        "next_action": update.get("next_action"),
        "iterations": update.get("iterations"),
        "used_tools": update.get("used_tools", []),
        "collections": update.get("collections", []),
        "context_chars": len(str(update.get("context", ""))),
        "answer_chars": len(str(update.get("answer", ""))),
        "changed_keys": sorted(update.keys()),
    }

def run_graph_with_trace(app: Any, initial_state: dict[str, Any]) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    final_state = dict(initial_state)
    trace: list[dict[str, Any]] = []
    for event in app.stream(initial_state, stream_mode="updates"):
        if not isinstance(event, dict):
            continue
        for node_name, update in event.items():
            if not isinstance(update, dict):
                continue
            final_state.update(update)
            trace.append(summarize_update(node_name, update))
    return final_state, trace

def build_retrieval_rows_for_report(mode: str, query: str, n_results: int = 8) -> list[dict[str, Any]]:
    apply_test_adapter(mode)
    rows = query_hanja_test_collections(query, n_results=n_results)
    return [
        {
            "rank": index,
            "id": row["id"],
            "test_collection": row["test_collection"],
            "distance": round(row["distance"], 4),
            "hanja": row["metadata"].get("hanja"),
            "hangul": row["metadata"].get("hangul"),
            "meaning": row["metadata"].get("sound_meaning"),
            "strokes": row["metadata"].get("strokes"),
            "sound_ohaeng": row["metadata"].get("sound_ohaeng"),
            "resource_ohaeng": row["metadata"].get("resource_ohaeng"),
            "source": row["metadata"].get("source"),
        }
        for index, row in enumerate(rows, 1)
    ]

def run_naming_graph_once(mode: str, query: str) -> dict[str, Any]:
    if not real_api_key or real_api_key == "sk-test-not-used":
        raise RuntimeError("GPT 실행에는 실제 OPENAI_API_KEY가 필요합니다. 설정 셀에서 키를 입력하고 RUN_LLM=True로 설정한 후 다시 실행하십시오.")
    apply_test_adapter(mode)
    app = naming_graph.build_graph()
    started = time.time()
    initial_state = build_initial_state(query)
    result, trace = run_graph_with_trace(app, initial_state)
    retrieval_rows = build_retrieval_rows_for_report(mode, query, n_results=8)
    return {
        "mode": mode,
        "query": query,
        "collections": TEST_MODES[mode],
        "elapsed_sec": round(time.time() - started, 2),
        "iterations": result.get("iterations"),
        "used_tools": result.get("used_tools", []),
        "trace": trace,
        "retrieval_rows": retrieval_rows,
        "answer": result.get("answer", ""),
        "context_preview": result.get("context", "")[:2000],
    }

print("naming_graph 실행 함수 준비 완료")


In [ ]:
# 8-0. 비교 결과 초기화
# RUN_LLM=False: ChromaDB 검색 근거와 연결 상태만 검증합니다.
# RUN_LLM=True: naming_graph.py 파이프라인을 통해 GPT 답변 생성까지 수행합니다.
# 8-1, 8-2, 8-3 셀을 개별 실행하여 DB 구성별 결과를 독립적으로 확인합니다.
MODE_DISPLAY_NAMES = {
    "Baseline": "A. 정제 완료 운영 후보군",
    "Expanded": "B. 확장 후보 포함군",
    "Hybrid": "C. 하이브리드 검토군",
}

MODE_OPERATIONAL_ROLES = {
    "Baseline": "유니코드, 뜻음, 획수, 발음오행, 자원오행이 모두 정제된 2,420건을 운영 기본값으로 검증합니다.",
    "Expanded": "정제 완료 2,420건에 확장 후보 6,564건을 합쳐 검색 폭을 넓혔을 때의 리스크를 검증합니다.",
    "Hybrid": "정제 완료군과 후보군을 분리 검색해 운영 보조 후보군으로 쓸 수 있는지 검증합니다.",
}

comparison_result: dict[str, Any] = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "question": QUESTION,
    "run_llm": RUN_LLM,
    "chroma_path": str(CHROMA_PATH),
    "index_report_path": str(INDEX_REPORT_PATH),
    "connection_report": connection_report,
    "preview_report": preview_report,
    "mode_display_names": MODE_DISPLAY_NAMES,
    "mode_operational_roles": MODE_OPERATIONAL_ROLES,
    "results": {},
}


def enrich_mode_result(mode: str, result: dict[str, Any]) -> dict[str, Any]:
    result["display_name"] = MODE_DISPLAY_NAMES.get(mode, mode)
    result["operational_role"] = MODE_OPERATIONAL_ROLES.get(mode, "")
    result["collections"] = result.get("collections") or TEST_MODES.get(mode, [])
    return result


def run_or_preview_mode(mode: str) -> dict[str, Any]:
    if mode not in TEST_MODES:
        raise ValueError(f"알 수 없는 테스트 모드입니다: {mode}")
    label = MODE_DISPLAY_NAMES.get(mode, mode)
    if RUN_LLM:
        print(f"{label} 실행 중: naming_graph 파이프라인 + GPT 답변 생성")
        return enrich_mode_result(mode, run_naming_graph_once(mode, QUESTION))

    print(f"{label} 실행 중: RUN_LLM=False, ChromaDB 검색 근거만 확인")
    apply_test_adapter(mode)
    return enrich_mode_result(mode, {
        "mode": mode,
        "query": QUESTION,
        "collections": TEST_MODES[mode],
        "trace": [],
        "retrieval_rows": build_retrieval_rows_for_report(mode, QUESTION, n_results=8),
        "answer": "RUN_LLM=False로 설정되어 GPT 답변은 생성하지 않았습니다. 검색 근거 미리보기를 확인하십시오.",
        "search_preview": format_hanja_rows(query_hanja_test_collections(QUESTION, n_results=5)),
    })


def print_mode_summary(mode: str) -> None:
    result = comparison_result["results"].get(mode)
    if not result:
        print(f"{MODE_DISPLAY_NAMES.get(mode, mode)}: 아직 실행되지 않았습니다.")
        return
    print(json.dumps({
        "mode": mode,
        "display_name": result.get("display_name", MODE_DISPLAY_NAMES.get(mode, mode)),
        "run_llm": RUN_LLM,
        "collections": result.get("collections", []),
        "retrieval_count": len(result.get("retrieval_rows", [])),
        "trace_nodes": [step.get("node") for step in result.get("trace", [])],
        "answer_chars": len(result.get("answer", "")),
    }, ensure_ascii=False, indent=2))

print(json.dumps({k: v for k, v in comparison_result.items() if k != "results"}, ensure_ascii=False, indent=2))


In [ ]:
# 8-1. A 후보군 실행 - 정제 완료 운영 후보군
# 정제 완료 데이터 2,420건만 포함한 hanja_base_test_col을 사용합니다.
comparison_result["results"]["Baseline"] = run_or_preview_mode("Baseline")
print_mode_summary("Baseline")


In [ ]:
# 8-2. B 후보군 실행 - 확장 후보 포함군
# 정제 완료 2,420건과 확장 후보 6,564건을 통합한 hanja_expanded_test_col을 사용합니다.
comparison_result["results"]["Expanded"] = run_or_preview_mode("Expanded")
print_mode_summary("Expanded")


In [ ]:
# 8-3. C 후보군 실행 - 하이브리드 검토군
# 정제 완료 컬렉션과 후보군 컬렉션을 분리 검색한 뒤 결과를 병합합니다.
comparison_result["results"]["Hybrid"] = run_or_preview_mode("Hybrid")
print_mode_summary("Hybrid")


In [ ]:
# 8-4. 저장 전 시각화 검증
# 개발자 검증: naming_graph 파이프라인에서 통과한 노드를 SVG 흐름도로 확인합니다.
# 결과 검토: 각 ChromaDB 구성별 질의응답 결과와 검색 근거를 화면에서 확인합니다.
from html import escape
from IPython.display import HTML, Markdown, SVG, display


def trace_nodes(result: dict[str, Any]) -> list[str]:
    nodes = [step.get("node", "unknown") for step in result.get("trace", [])]
    return nodes or ["검색 미리보기", "GPT 미실행"]


def mode_label(mode: str, result: dict[str, Any] | None = None) -> str:
    if result and result.get("display_name"):
        return result["display_name"]
    return MODE_DISPLAY_NAMES.get(mode, mode) if "MODE_DISPLAY_NAMES" in globals() else mode


def make_pipeline_svg(mode: str, result: dict[str, Any]) -> str:
    nodes = trace_nodes(result)
    label = mode_label(mode, result)
    width = max(760, 170 * len(nodes))
    height = 178
    colors = {
        "llm_router": "#2563eb",
        "internal_rag": "#059669",
        "generate": "#7c3aed",
        "clarify": "#d97706",
        "graph_db": "#0f766e",
        "sql_db": "#0891b2",
        "external_api": "#be123c",
    }
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#f8fafc" rx="10"/>',
        f'<text x="24" y="30" font-size="18" font-weight="700" fill="#111827">{escape(label)} 파이프라인 흐름</text>',
        f'<text x="24" y="54" font-size="12" fill="#475569">컬렉션: {escape(", ".join(result.get("collections", [])))}</text>',
        f'<text x="24" y="73" font-size="12" fill="#475569">역할: {escape(result.get("operational_role", ""))}</text>',
    ]
    start_x = 28
    y = 96
    box_w = 132
    box_h = 46
    gap = 38
    for index, node in enumerate(nodes):
        x = start_x + index * (box_w + gap)
        color = colors.get(node, "#64748b")
        parts.append(f'<rect x="{x}" y="{y}" width="{box_w}" height="{box_h}" rx="8" fill="{color}"/>')
        parts.append(f'<text x="{x + box_w / 2}" y="{y + 29}" text-anchor="middle" font-size="13" font-weight="700" fill="white">{escape(node)}</text>')
        if index < len(nodes) - 1:
            x1 = x + box_w + 8
            x2 = x + box_w + gap - 8
            mid = y + box_h / 2
            parts.append(f'<line x1="{x1}" y1="{mid}" x2="{x2}" y2="{mid}" stroke="#94a3b8" stroke-width="2"/>')
            parts.append(f'<polygon points="{x2},{mid} {x2 - 8},{mid - 5} {x2 - 8},{mid + 5}" fill="#94a3b8"/>')
    parts.append('</svg>')
    return "".join(parts)


def retrieval_table_html(rows: list[dict[str, Any]]) -> str:
    if not rows:
        return '<p style="color:#64748b;">검색 근거가 없습니다.</p>'
    header = ''.join(f'<th>{name}</th>' for name in ["순위", "한자", "음", "뜻", "획수", "발음오행", "자원오행", "출처", "컬렉션"])
    body = []
    for row in rows:
        body.append(
            '<tr>'
            f'<td>{row.get("rank")}</td>'
            f'<td><strong>{escape(str(row.get("hanja", "")))}</strong></td>'
            f'<td>{escape(str(row.get("hangul", "")))}</td>'
            f'<td>{escape(str(row.get("meaning", "")))}</td>'
            f'<td>{escape(str(row.get("strokes", "")))}</td>'
            f'<td>{escape(str(row.get("sound_ohaeng", "")))}</td>'
            f'<td>{escape(str(row.get("resource_ohaeng", "")))}</td>'
            f'<td>{escape(str(row.get("source", "")))}</td>'
            f'<td>{escape(str(row.get("test_collection", "")))}</td>'
            '</tr>'
        )
    return (
        '<table style="border-collapse:collapse;width:100%;font-size:13px;">'
        '<thead><tr style="background:#f1f5f9;">' + header + '</tr></thead>'
        '<tbody>' + ''.join(body) + '</tbody></table>'
        '<style>td,th{border:1px solid #cbd5e1;padding:6px 8px;text-align:left;vertical-align:top;} th{font-weight:700;color:#0f172a;}</style>'
    )


def display_mode_result(mode: str, result: dict[str, Any]) -> None:
    label = mode_label(mode, result)
    display(SVG(make_pipeline_svg(mode, result)))
    display(Markdown(f"### {label} 사용자 QA 미리보기"))
    display(Markdown(f"**질문**\n\n{result.get('query', QUESTION)}"))
    display(Markdown("**답변**"))
    display(Markdown(result.get("answer", "")))
    display(Markdown("**ChromaDB 검색 근거**"))
    display(HTML(retrieval_table_html(result.get("retrieval_rows", []))))
    if result.get("trace"):
        display(Markdown("**개발자용 trace 요약**"))
        display(HTML('<pre style="white-space:pre-wrap;background:#0f172a;color:#e2e8f0;padding:12px;border-radius:8px;">' + escape(json.dumps(result.get("trace", []), ensure_ascii=False, indent=2)) + '</pre>'))


executed_modes = list(comparison_result.get("results", {}).keys())
display(Markdown("## 저장 전 비교 결과 확인"))
if not executed_modes:
    display(Markdown("아직 실행된 모드가 없습니다. 8-1, 8-2, 8-3 셀 중 확인할 모드를 먼저 실행하세요."))
else:
    display(Markdown("8-1~8-3 실행 결과를 화면에서 확인합니다. 9번 저장 셀을 실행하지 않아도 본 셀에서 파이프라인 흐름, 사용자 QA, 검색 근거를 확인할 수 있습니다."))
    for mode, result in comparison_result.get("results", {}).items():
        display_mode_result(mode, result)
print("실행된 결과 모드:", [mode_label(mode, result) for mode, result in comparison_result.get("results", {}).items()])


In [ ]:
# 9. 운영 근거 보고서 및 비교 산출물 저장
# 목적: 정제 완료 2,420건을 운영용 hanja_col 기본값으로 사용하는 근거를 보고서 형태로 저장합니다.
# 기존 naming_graph 파이프라인과 시각화 구조는 유지하며, 산출물은 운영 판단 근거 중심으로 정리합니다.
import csv
import re

RESULT_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
json_path = RESULT_DIR / f"hanja_operational_candidate_comparison_{stamp}_full.json"
md_path = RESULT_DIR / f"hanja_operational_candidate_comparison_{stamp}_report.md"
summary_csv_path = RESULT_DIR / f"hanja_operational_candidate_comparison_{stamp}_summary.csv"

BASE_DOC_PATH = ROOT / "data" / "processed" / "hanja_documents.json"
CANDIDATE_DOC_PATH = ROOT / "data" / "processed" / "hanja2_candidate_documents.json"


def load_hanja_records(path: Path) -> list[dict[str, Any]]:
    return json.loads(path.read_text(encoding="utf-8"))


base_records = load_hanja_records(BASE_DOC_PATH)
candidate_records = load_hanja_records(CANDIDATE_DOC_PATH)
base_hanja = {item.get("metadata", {}).get("hanja") for item in base_records if item.get("metadata", {}).get("hanja")}
candidate_hanja = {item.get("metadata", {}).get("hanja") for item in candidate_records if item.get("metadata", {}).get("hanja")}
comparison_total_count = len(base_records) + len(candidate_records)

MODE_DISPLAY_NAMES = globals().get("MODE_DISPLAY_NAMES", {
    "Baseline": "A. 정제 완료 운영 후보군",
    "Expanded": "B. 확장 후보 포함군",
    "Hybrid": "C. 하이브리드 검토군",
})
MODE_OPERATIONAL_ROLES = globals().get("MODE_OPERATIONAL_ROLES", {
    "Baseline": "유니코드, 뜻음, 획수, 발음오행, 자원오행이 모두 정제된 2,420건을 운영 기본값으로 검증합니다.",
    "Expanded": "정제 완료 2,420건에 확장 후보 6,564건을 합쳐 검색 폭을 넓혔을 때의 리스크를 검증합니다.",
    "Hybrid": "정제 완료군과 후보군을 분리 검색해 운영 보조 후보군으로 쓸 수 있는지 검증합니다.",
})

MODE_DATASET_SUMMARY = {
    "Baseline": {
        "label": MODE_DISPLAY_NAMES["Baseline"],
        "count": len(base_records),
        "data": "hanja_documents.json",
        "meaning": "정제 완료 운영 기본 후보군",
        "operational_position": "운영 기본값 권장",
    },
    "Expanded": {
        "label": MODE_DISPLAY_NAMES["Expanded"],
        "count": comparison_total_count,
        "data": "hanja_documents.json + hanja2_candidate_documents.json",
        "meaning": "정제 완료군과 확장 후보군을 하나로 합친 실험군",
        "operational_position": "운영 기본값으로는 보류, 확장 검토용",
    },
    "Hybrid": {
        "label": MODE_DISPLAY_NAMES["Hybrid"],
        "count": f"{len(base_records)} + {len(candidate_records)}",
        "data": "hanja_base_test_col + hanja_candidate_test_col",
        "meaning": "정제 완료군과 후보군을 분리 검색하는 검토군",
        "operational_position": "보조 후보군 검토용",
    },
}

REQUIRED_RETRIEVAL_FIELDS = ["hanja", "hangul", "meaning", "strokes", "sound_ohaeng", "resource_ohaeng", "source", "test_collection"]


def extract_hanja_chars(text: str) -> set[str]:
    return set(re.findall(r"[一-鿿]", text or ""))


def available_hanja_for_mode(mode: str) -> set[str]:
    if mode == "Baseline":
        return set(base_hanja)
    return set(base_hanja) | set(candidate_hanja)


def count_filled_metadata(rows: list[dict[str, Any]]) -> tuple[int, int, float]:
    total = len(rows) * len(REQUIRED_RETRIEVAL_FIELDS)
    filled = sum(
        1
        for row in rows
        for field in REQUIRED_RETRIEVAL_FIELDS
        if row.get(field) not in (None, "", "미확인", "검증대기")
    )
    ratio = 0.0 if total == 0 else filled / total
    return filled, total, ratio


def clamp(value: float, low: float = 0.0, high: float = 100.0) -> float:
    return max(low, min(high, value))


def evaluate_operational_fit(mode: str, result: dict[str, Any]) -> dict[str, Any]:
    rows = result.get("retrieval_rows", []) or []
    answer = result.get("answer", "") or ""
    answer_chars = extract_hanja_chars(answer) - extract_hanja_chars(result.get("query", QUESTION))
    top_evidence_hanja = {row.get("hanja") for row in rows if row.get("hanja")}
    available_hanja = available_hanja_for_mode(mode)
    db_unsupported = sorted(answer_chars - available_hanja)
    top_evidence_not_shown = sorted(answer_chars - top_evidence_hanja - extract_hanja_chars(result.get("query", QUESTION)))
    candidate_rows = [row for row in rows if str(row.get("source", "")).startswith("hanja2_candidate")]
    base_rows = [row for row in rows if not str(row.get("source", "")).startswith("hanja2_candidate")]
    filled, total, metadata_ratio = count_filled_metadata(rows)
    trace_nodes = [step.get("node") for step in result.get("trace", [])]
    has_core_pipeline = all(node in trace_nodes for node in ["llm_router", "internal_rag", "generate"]) if RUN_LLM else False
    has_model_error = "모델 오류" in answer or "[모델 오류" in answer

    if mode == "Baseline":
        curation_score = 35
        risk_control_score = 20
    elif mode == "Hybrid":
        curation_score = 24 if candidate_rows else 30
        risk_control_score = 13 if candidate_rows else 16
    else:
        curation_score = 22 if candidate_rows else 28
        risk_control_score = 9 if candidate_rows else 14

    metadata_score = round(metadata_ratio * 10, 2)
    if RUN_LLM:
        grounded_score = 25 if not db_unsupported else max(0, 25 - len(db_unsupported) * 5)
        pipeline_score = 20 if has_core_pipeline and not has_model_error else (12 if trace_nodes and not has_model_error else 6)
    else:
        grounded_score = 15
        pipeline_score = 10

    total_score = round(clamp(curation_score + metadata_score + grounded_score + pipeline_score + risk_control_score), 2)

    if mode == "Baseline":
        recommendation = "운영 기본값 권장"
        verdict = "정제 완료 2,420건만 사용하므로 설명 가능성과 오행 매핑 안정성이 가장 높다."
    elif mode == "Expanded":
        recommendation = "운영 기본값 보류"
        verdict = "검색 범위는 넓지만 후보군이 한 컬렉션에 섞여 자동 추천 결과의 통제 리스크가 커진다."
    else:
        recommendation = "보조 후보군 검토"
        verdict = "정제 완료군과 후보군을 분리할 수 있어 실험용으로 유용하지만, 후보군 채택 정책이 추가로 필요하다."

    return {
        "mode": mode,
        "label": result.get("display_name", MODE_DISPLAY_NAMES.get(mode, mode)),
        "total_score": total_score,
        "max_score": 100,
        "recommendation": recommendation,
        "verdict": verdict,
        "scores": {
            "curation_certainty_35": curation_score,
            "metadata_completeness_10": metadata_score,
            "db_groundedness_25": grounded_score,
            "pipeline_stability_20": pipeline_score,
            "candidate_risk_control_20": risk_control_score,
        },
        "metrics": {
            "retrieved_count": len(rows),
            "base_evidence_count": len(base_rows),
            "candidate_evidence_count": len(candidate_rows),
            "metadata_completeness": round(metadata_ratio, 4),
            "answer_hanja_count": len(answer_chars),
            "db_unsupported_hanja": db_unsupported,
            "top_evidence_not_shown_hanja": top_evidence_not_shown,
            "trace_nodes": trace_nodes,
            "elapsed_sec": result.get("elapsed_sec"),
            "used_tools": result.get("used_tools", []),
        },
    }


def validate_operational_report(fit_cards: dict[str, Any]) -> dict[str, Any]:
    issues = []
    for mode in ["Baseline", "Expanded", "Hybrid"]:
        if mode not in fit_cards:
            issues.append(f"비교 모드 누락: {mode}")
    for mode, card in fit_cards.items():
        if card["total_score"] < 0 or card["total_score"] > 100:
            issues.append(f"{mode}: 운영 적합성 점수 범위 초과")
        if card["metrics"]["db_unsupported_hanja"]:
            issues.append(f"{mode}: DB 전체 기준으로도 없는 한자가 답변에 포함됨")
    return {"ok": not issues, "issues": issues}


def md_table(rows: list[list[Any]]) -> list[str]:
    header = rows[0]
    lines = ["| " + " | ".join(map(str, header)) + " |", "| " + " | ".join(["---"] * len(header)) + " |"]
    for row in rows[1:]:
        lines.append("| " + " | ".join(map(lambda value: str(value).replace("\n", " "), row)) + " |")
    return lines


def md_retrieval_table(rows: list[dict[str, Any]]) -> list[str]:
    table = [["순위", "한자", "음", "뜻", "획수", "발음오행", "자원오행", "출처", "컬렉션"]]
    for row in rows[:8]:
        table.append([
            row.get("rank"), row.get("hanja"), row.get("hangul"), row.get("meaning"), row.get("strokes"),
            row.get("sound_ohaeng"), row.get("resource_ohaeng"), row.get("source"), row.get("test_collection"),
        ])
    return md_table(table)


fit_cards = {
    mode: evaluate_operational_fit(mode, result)
    for mode, result in comparison_result.get("results", {}).items()
}
report_validation = validate_operational_report(fit_cards)
ranking = sorted(fit_cards.values(), key=lambda card: card["total_score"], reverse=True)
recommended_mode = "Baseline"

operational_argument = {
    "comparison_total_count": comparison_total_count,
    "base_count": len(base_records),
    "candidate_count": len(candidate_records),
    "recommended_mode": recommended_mode,
    "recommended_label": MODE_DISPLAY_NAMES[recommended_mode],
    "core_claim": "전체 후보를 모두 운영 DB로 쓰지 않고, 유니코드/뜻음/획수/발음오행/자원오행이 정제된 2,420건을 운영 기본 hanja_col로 사용하는 것이 타당하다.",
    "reasoning": [
        "이름 추천은 검색 범위보다 뜻음, 획수, 발음오행, 자원오행의 일관된 근거가 더 중요하다.",
        "2,420건은 운영용 설명 문장과 metadata가 완성된 정제 완료군이다.",
        "6,564건은 확장 후보군으로 보관할 수 있지만 자동 추천 기본 DB에 섞으면 후보군 통제와 검수 책임이 커진다.",
        "Hybrid 구조는 보조 후보 검토에는 유용하지만 운영 기본값으로 쓰려면 후보군 채택 정책과 후처리 검증이 추가로 필요하다.",
    ],
}

comparison_result["operational_argument"] = operational_argument
comparison_result["operational_fit_cards"] = fit_cards
comparison_result["report_validation"] = report_validation
comparison_result["ranking"] = ranking
comparison_result["recommended_mode"] = recommended_mode
comparison_result["recommended_label"] = MODE_DISPLAY_NAMES[recommended_mode]

with summary_csv_path.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "mode", "label", "total_score", "recommendation", "dataset_count", "retrieved_count",
            "base_evidence_count", "candidate_evidence_count", "metadata_completeness",
            "db_unsupported_hanja_count", "top_evidence_not_shown_hanja_count", "elapsed_sec", "verdict",
        ],
    )
    writer.writeheader()
    for mode, card in fit_cards.items():
        metrics = card["metrics"]
        writer.writerow({
            "mode": mode,
            "label": card["label"],
            "total_score": card["total_score"],
            "recommendation": card["recommendation"],
            "dataset_count": MODE_DATASET_SUMMARY[mode]["count"],
            "retrieved_count": metrics["retrieved_count"],
            "base_evidence_count": metrics["base_evidence_count"],
            "candidate_evidence_count": metrics["candidate_evidence_count"],
            "metadata_completeness": metrics["metadata_completeness"],
            "db_unsupported_hanja_count": len(metrics["db_unsupported_hanja"]),
            "top_evidence_not_shown_hanja_count": len(metrics["top_evidence_not_shown_hanja"]),
            "elapsed_sec": metrics["elapsed_sec"],
            "verdict": card["verdict"],
        })

json_path.write_text(json.dumps(comparison_result, ensure_ascii=False, indent=2), encoding="utf-8")

md_lines = [
    "# 한자 운영 DB 후보군 비교 보고서",
    "",
    f"- 질문: {QUESTION}",
    f"- GPT 호출 여부: {RUN_LLM}",
    f"- 테스트 Chroma 경로: `{CHROMA_PATH}`",
    f"- 요약 CSV: `{summary_csv_path.name}`",
    "",
    "## 1. 핵심 결론",
    "",
    f"**운영 기본값은 {MODE_DISPLAY_NAMES['Baseline']}이 적합합니다.**",
    "",
    operational_argument["core_claim"],
    "",
    "이 결론은 전체 후보를 버린다는 의미가 아니라, 자동 작명 추천의 기본 DB는 정제 완료군으로 고정하고 확장 후보군은 검토/보조 계층으로 분리해야 한다는 의미입니다.",
    "",
    "## 2. 왜 2,420개만 운영용 hanja_col로 쓰는가",
    "",
]
md_lines.extend(md_table([
    ["구분", "건수", "역할", "운영 판단"],
    ["정제 완료 한자", len(base_records), "유니코드, 뜻음, 획수, 발음오행, 자원오행이 정리된 운영 기본 후보군", "운영 기본값"],
    ["확장 후보 한자", len(candidate_records), "원본에서 추가 확보한 후보군", "자동 운영 기본값 보류"],
    ["비교 기준 전체", comparison_total_count, "정제 완료군 + 확장 후보군", "실험/검토 기준"],
]))
md_lines.extend(["", "운영 기본값을 2,420건으로 두는 근거는 다음과 같습니다.", ""])
for reason in operational_argument["reasoning"]:
    md_lines.append(f"- {reason}")

md_lines.extend(["", "## 3. A/B/C 테스트 구성", ""])
md_lines.extend(md_table([
    ["구분", "데이터", "건수", "의미", "운영 위치"],
    *[
        [summary["label"], summary["data"], summary["count"], summary["meaning"], summary["operational_position"]]
        for summary in MODE_DATASET_SUMMARY.values()
    ],
]))

md_lines.extend(["", "## 4. 실행 결과 요약", ""])
md_lines.extend(md_table([
    ["구분", "운영 적합성", "권고", "검색 근거", "후보군 근거", "DB 미존재 한자", "상위 근거 미노출", "판정"],
    *[
        [
            card["label"],
            f"{card['total_score']}/100",
            card["recommendation"],
            card["metrics"]["retrieved_count"],
            card["metrics"]["candidate_evidence_count"],
            len(card["metrics"]["db_unsupported_hanja"]),
            len(card["metrics"]["top_evidence_not_shown_hanja"]),
            card["verdict"],
        ]
        for card in fit_cards.values()
    ],
]))
md_lines.extend([
    "",
    f"- 보고서 검증 결과: {'통과' if report_validation['ok'] else '확인 필요'}",
    f"- 최종 권장안: **{MODE_DISPLAY_NAMES[recommended_mode]}**",
])
if report_validation["issues"]:
    md_lines.extend(["- 검증 이슈:"] + [f"  - {issue}" for issue in report_validation["issues"]])

md_lines.extend(["", "## 5. 모드별 상세", ""])
for mode, result in comparison_result.get("results", {}).items():
    card = fit_cards[mode]
    metrics = card["metrics"]
    md_lines.extend([
        f"### {card['label']}",
        "",
        f"- 역할: {MODE_OPERATIONAL_ROLES.get(mode, '')}",
        f"- 사용 컬렉션: {', '.join(result.get('collections', []))}",
        f"- 운영 적합성: {card['total_score']}/100",
        f"- 권고: {card['recommendation']}",
        f"- 판정: {card['verdict']}",
        f"- DB 전체 기준 미존재 한자: {metrics['db_unsupported_hanja'] or '없음'}",
        f"- 상위 8개 검색근거 미노출 한자: {metrics['top_evidence_not_shown_hanja'] or '없음'}",
        "",
        "#### ChromaDB 검색 근거",
    ])
    md_lines.extend(md_retrieval_table(result.get("retrieval_rows", [])))
    md_lines.extend(["", "#### 사용자 QA 결과", ""])
    if RUN_LLM:
        md_lines.append(result.get("answer", ""))
    else:
        md_lines.extend(["```text", result.get("search_preview", ""), "```"])
    if result.get("trace"):
        md_lines.extend(["", "#### 기존 파이프라인 trace", "", "```json", json.dumps(result.get("trace", []), ensure_ascii=False, indent=2), "```"])
    md_lines.append("")

md_lines.extend([
    "## 6. 반론 대응 문장",
    "",
    "> 전체 원본 한자 후보가 9천여 개 수준인데 운영 DB가 2,420개인 이유는 누락이 아니라 정제 기준 때문입니다. 작명 추천은 한자 수를 많이 확보하는 것보다 유니코드, 뜻음, 획수, 발음오행, 자원오행이 일관되게 검증된 데이터를 사용하는 것이 중요합니다. 따라서 정제 완료 2,420건을 운영용 hanja_col의 기본값으로 두고, 나머지 확장 후보군은 별도 후보/검토 계층으로 유지하는 것이 안정적인 선택입니다.",
    "",
    "> B/C 후보군은 검색 폭을 넓히는 장점이 있지만 자동 추천 기본 DB로 바로 섞기에는 후보군 통제, 설명 책임, 후처리 검증 정책이 더 필요합니다. 그러므로 현재 운영 기준은 A 후보군을 기본값으로 고정하는 것이 타당합니다.",
])

md_path.write_text("\n".join(md_lines), encoding="utf-8")

print("JSON 저장:", json_path)
print("MD 보고서 저장:", md_path)
print("요약 CSV 저장:", summary_csv_path)
print("보고서 검증:", "통과" if report_validation["ok"] else "확인 필요", report_validation["issues"])
print("최종 권장안:", MODE_DISPLAY_NAMES[recommended_mode])
